# Identified works → local Iceberg table

Pulls the matcher-relevant columns of every document in the prod `works-identified` index into a local Iceberg table. Reruns are no-ops once the table is populated.

In [ ]:
import time
import pyarrow as pa
from pyiceberg.io.pyarrow import schema_to_pyarrow
from pyiceberg.schema import Schema
from pyiceberg.table.sorting import SortField, SortOrder
from pyiceberg.transforms import IdentityTransform
from pyiceberg.types import IntegerType, ListType, NestedField, StringType

from adapters.utils.iceberg import LocalIcebergTableConfig, get_local_table
from core.source import ElasticSource
from utils.elasticsearch import get_client

PIPELINE_DATE = "2025-10-02"
INDEX = f"works-identified-{PIPELINE_DATE}"

SCHEMA = Schema(
    NestedField(1, "id", StringType(), required=True),
    NestedField(2, "source_identifier_type", StringType(), required=True),
    NestedField(3, "source_identifier_value", StringType(), required=True),
    NestedField(4, "version", IntegerType(), required=True),
    NestedField(5, "type", StringType(), required=True),
    NestedField(6, "merge_candidate_ids", ListType(7, StringType(), element_required=True), required=True),
    NestedField(8, "merge_candidate_reasons", ListType(9, StringType(), element_required=True), required=True),
    NestedField(10, "source_modified_time", StringType(), required=False),
    NestedField(11, "content", StringType(), required=False),
)
table = get_local_table(
    LocalIcebergTableConfig(
        table_name="works_identified",
        namespace="matcher",
        db_name="matcher_catalog",
        iceberg_schema=SCHEMA,
        sort_order=SortOrder(SortField(source_id=1, transform=IdentityTransform())),
    )
)
ARROW_SCHEMA = schema_to_pyarrow(table.schema())
populated = table.scan(limit=1).to_arrow().num_rows > 0
print("table populated:", populated)

In [ ]:
def to_row(doc: dict) -> dict:
    state = doc["state"]
    candidates = state.get("mergeCandidates", [])
    return {
        "id": state["canonicalId"],
        "source_identifier_type": state["sourceIdentifier"]["identifierType"]["id"],
        "source_identifier_value": state["sourceIdentifier"]["value"],
        "version": doc["version"],
        "type": doc["type"],
        "merge_candidate_ids": [c["id"]["canonicalId"] for c in candidates],
        "merge_candidate_reasons": [c["reason"] for c in candidates],
        "source_modified_time": state.get("sourceModifiedTime"),
        "content": None,
    }


if not populated:
    es = get_client("read_only", PIPELINE_DATE, "public")
    source = ElasticSource(
        es_client=es,
        index_name=INDEX,
        query={"match_all": {}},
        fields=["state.canonicalId", "state.sourceIdentifier", "state.mergeCandidates", "state.sourceModifiedTime", "version", "type"],
    )
    started = time.time()
    rows, total = [], 0
    for doc in source.stream_raw():
        rows.append(to_row(doc))
        if len(rows) == 200_000:
            table.append(pa.Table.from_pylist(rows, schema=ARROW_SCHEMA).sort_by("id"))
            total += len(rows)
            rows = []
            print(f"{total:>9,} rows  {time.time() - started:6.0f}s")
    if rows:
        table.append(pa.Table.from_pylist(rows, schema=ARROW_SCHEMA).sort_by("id"))
        total += len(rows)
    print(f"done: {total:,} rows in {time.time() - started:.0f}s")

In [ ]:
started = time.time()
ids = table.scan(selected_fields=("id",)).to_arrow()
print(f"{ids.num_rows:,} rows, {len(set(ids.column('id').to_pylist())):,} distinct ids, read in {time.time() - started:.1f}s")